In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Finansal Analiz

Amaç: Bir analistin, bir şirketin mali raporundan spesifik rakamları (gelir, kar vb.) sorgulaması.

In [ ]:
# Gerekli kütüphaneler: pip install sentence-transformers scikit-learn numpy
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- BİLGİ KAYNAĞI ---
KNOWLEDGE_BASE = """
Finansal Özet - 2024 2. Çeyrek:
TeknoYatırım A.Ş. olarak 2024 yılının ikinci çeyreğinde güçlü bir performans sergiledik. Toplam gelirimiz geçen yılın aynı dönemine göre %25 artarak 150 Milyon TL'ye ulaşmıştır. Net kar marjımız %18 olarak gerçekleşmiş ve 27 Milyon TL net kar elde edilmiştir.

Departman Harcamaları ve Yatırımlar:
Bu çeyrekte Ar-Ge harcamalarına 15 Milyon TL yatırım yapılmıştır. Bu yatırım, yeni nesil yapay zeka algoritmaları geliştirmek için kullanılmıştır. Pazarlama ve satış giderleri ise toplam 25 Milyon TL olarak kaydedilmiştir.
"""

# --- MODÜLER FONKSİYONLAR ---
def chunk_text(text: str) -> list[str]:
    return [p.strip() for p in text.split('\n\n') if p.strip()]

def get_embeddings(chunks, model):
    return model.encode(chunks)

def find_most_similar_chunks(query, embeddings, chunks, model, top_k=1):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    return [chunks[i] for i in top_k_indices]

def generate_answer(query, context):
    prompt = f"Rapor Özeti: {context}\n\nSoru: {query}\n\nCevap:"
    print("--- LLM'e Gönderilen Nihai Prompt ---\n", prompt)
    mock_response = f"Mali rapora göre, '{query}' sorunuzun cevabı şudur: {context}"
    return mock_response

# --- ANA PROGRAM AKIŞI ---
if __name__ == "__main__":
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    user_query = "Şirketin ikinci çeyrekteki geliri ve net karı ne kadar oldu?"
    
    print("--- Finansal Analiz RAG Demosu ---")
    print(f"Kullanıcı Sorusu: '{user_query}'\n")

    chunks = chunk_text(KNOWLEDGE_BASE)
    print(f"--- 1. Parçalama Sonucu: {len(chunks)} adet rapor bölümü bulundu. ---\n")
    
    document_embeddings = get_embeddings(chunks, embedding_model)
    
    context_chunks = find_most_similar_chunks(user_query, document_embeddings, chunks, embedding_model)
    context_for_llm = "\n".join(context_chunks)
    print("--- 2. Arama Sonucu: En Alakalı Rapor Bölümü Bulundu ---\n", context_for_llm, "\n")
    
    final_answer = generate_answer(user_query, context_for_llm)
    print("\n--- 3. Nihai Cevap Üretildi ---")
    print(final_answer)